In [1]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.decomposition import TruncatedSVD
import os
import wandb
from scipy.stats import uniform, loguniform, randint
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from util.preprocessing import TweetPreprocessor 
from util.logger import wandb_log_search_results, wandb_save_model
from util.plot import plot_all_heatmaps

# Consts

In [3]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')

# Dataset

In [4]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))
train_df = pd.concat([train_df, val_df], ignore_index=True)

In [5]:
x_train, y_train = train_df["text"], train_df["gender_label"]

# Initiate pipeline

In [6]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])

# Init wandb

In [7]:
wandbToken = os.getenv("WANDB_TOKEN")
if not wandbToken:
    raise ValueError("Please set the WANDB_TOKEN environment variable to log results to Weights & Biases.")
wandb.login(key=wandbToken)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/user/.netrc
wandb: Currently logged in as: qgurulev (who-wrote-it-nlp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Randomized search

In [8]:
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=int(os.getenv("RANDOM_SEED", 880055535)),
)

## TF-idf 

In [9]:
texts = TweetPreprocessor().fit_transform(x_train) # type: ignore

### Word

In [ ]:
ngram_ranges_word = [(1, 2), (1, 3)]

rnd_params_tfidf_word = {
    "tfidf_word__use_idf": [True, False],
    "tfidf_word__sublinear_tf": [True, False],
    "tfidf_word__norm": ["l1", "l2"],
    "tfidf_word__max_df": uniform(0.6, 0.4),
    "tfidf_word__min_df": uniform(0.001, 0.4),
    "tfidf_word__max_features": randint(5000, 120000),
    "tfidf_word__ngram_range": ngram_ranges_word,
}
rnd_params_tfidf_word

{'tfidf_word__use_idf': [True, False],
 'tfidf_word__max_features': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x72f34f823380>}

In [11]:
rnd_search_tfidf_word = RandomizedSearchCV(
    Pipeline([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ]),
    rnd_params_tfidf_word, 
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_search_tfidf_word.fit(texts, y_train)

Fitting 4 folds for each of 500 candidates, totalling 2000 fits


In [ ]:
rnd_params_after_tfidf_word = rnd_search_tfidf_word.best_params_
rnd_params_after_tfidf_word

{'tfidf_word__use_idf': True}

In [ ]:
wandb_log_search_results(rnd_search_tfidf_word, "hyperparams_rnd_search_word", "hyperparams_rnd_search")

wandb: ERROR Control-C detected -- Run data was not synced


KeyboardInterrupt: 

In [ ]:
plot_all_heatmaps(random_search=rnd_search_tfidf_word)

Found 1 hyperparameters: ['param_tfidf_word__use_idf']
Need at least 2 hyperparameters to build heatmaps.


### Char

In [ ]:
ngram_ranges_char = [(2, 4), (3, 5), (4, 6)]

rnd_params_tfidf_char = {
    "tfidf_char__use_idf": [True, False],
    "tfidf_char__sublinear_tf": [True, False],
    "tfidf_char__norm": ["l1", "l2"],
    "tfidf_char__max_df": uniform(0.6, 0.4),
    "tfidf_char__min_df": uniform(0.001, 0.4),
    "tfidf_char__max_features": randint(5000, 120000),
    "tfidf_char__ngram_range": ngram_ranges_char,
}
rnd_params_tfidf_char

In [ ]:
rnd_search_tfidf_char = RandomizedSearchCV(
    Pipeline([
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
        ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ]),
    rnd_params_tfidf_char,
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_search_tfidf_char.fit(texts, y_train)

In [ ]:
rnd_params_after_tfidf_char = rnd_search_tfidf_char.best_params_
rnd_params_after_tfidf_char

In [ ]:
wandb_log_search_results(rnd_search_tfidf_word, "hyperparams_rnd_search_char", "hyperparams_rnd_search")

## SVD

In [ ]:
rnd_svd_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
rnd_merged_params_for_svd = {
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_word.items()},
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_char.items()},
}
print(rnd_merged_params_for_svd)
rnd_svd_pipeline.set_params(**rnd_merged_params_for_svd)

In [ ]:
rnd_svd_params = {
    "svd__n_components": randint(100, 500),
}
rnd_svd_params

In [ ]:
rnd_svd_search = RandomizedSearchCV(
    rnd_svd_pipeline,
    rnd_svd_params, 
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_svd_search.fit(texts, y_train)

In [ ]:
rnd_params_after_svd = rnd_svd_search.best_params_
rnd_params_after_svd

In [ ]:
wandb_log_search_results(rnd_search_tfidf_word, "hyperparams_rnd_search_svd", "hyperparams_rnd_search")

## CLF

In [ ]:
rnd_clf_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
rnd_merged_params_for_clf = {
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_word.items()},
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_char.items()},
    **rnd_params_after_svd,
}
print(rnd_merged_params_for_clf)
rnd_clf_pipeline.set_params(**rnd_merged_params_for_clf)

In [ ]:
rnd_clf_params = {
    "clf__C": loguniform(1e-3, 1e3),
}
rnd_clf_params

In [ ]:
rnd_clf_search = RandomizedSearchCV(
    rnd_clf_pipeline,
    rnd_clf_params,
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_clf_search.fit(texts, y_train)

In [ ]:
wandb_log_search_results(rnd_search_tfidf_word, "hyperparams_rnd_search_clf", "hyperparams_rnd_search")

In [ ]:
wandb_save_model(
    rnd_clf_search.best_estimator_,
    os.path.join(MODEL_DIR, "best_model_after_random_search"),
)

# Grid search

In [ ]:
best_rnd_params = rnd_clf_search.best_params_

## TF-idf

### Word

In [ ]:
def make_range(value, pct=0.05, cast_int=False):
    factors = [1 - pct, 1, 1 + pct]

    candidates = [value * f for f in factors]

    if cast_int:
        candidates = [int(round(c)) for c in candidates]

    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)

    return uniq

In [ ]:
grid_tfidf_params_word = {
    "tfidf_word__use_idf": [best_rnd_params["features__tfidf_word__use_idf"]],
    "tfidf_word__sublinear_tf": [best_rnd_params["features__tfidf_word__sublinear_tf"]],
    "tfidf_word__norm": [best_rnd_params["features__tfidf_word__norm"]],
    "tfidf_word__ngram_range": [best_rnd_params["features__tfidf_word__ngram_range"]],
    "tfidf_word__max_df": make_range(best_rnd_params["features__tfidf_word__max_df"]),
    "tfidf_word__min_df": make_range(best_rnd_params["features__tfidf_word__min_df"]),
    "tfidf_word__max_features": make_range(
        best_rnd_params["tfidf_word__max_features"], cast_int=True
    ),
}
grid_tfidf_params_word

In [ ]:
grid_tfidf_search_word = GridSearchCV(
    Pipeline([
        ("tfidf_char", TfidfVectorizer(analyzer="word")), # type: ignore
        ("clf", LinearSVC(
            C=best_rnd_params["clf__C"],
            random_state=int(os.getenv("RANDOM_SEED", 880055535)),
        )),
    ]),
    grid_tfidf_params_word,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

In [ ]:
grid_tfidf_search_word.fit(texts, y_train)

In [ ]:
best_tfidf_word = grid_tfidf_search_word.best_params_
best_tfidf_word

In [ ]:
wandb_log_search_results(grid_tfidf_search_word, "hyperparams_grid_search_word", "hyperparams_grid_search")

### Char

In [ ]:
grid_tfidf_params_char = {
    "tfidf_char__use_idf": [best_rnd_params["features__tfidf_char__use_idf"]],
    "tfidf_char__sublinear_tf": [best_rnd_params["features__tfidf_char__sublinear_tf"]],
    "tfidf_char__norm": [best_rnd_params["features__tfidf_char__norm"]],
    "tfidf_char__ngram_range": [best_rnd_params["features__tfidf_char__ngram_range"]],
    "tfidf_char__max_df": make_range(best_rnd_params["features__tfidf_char__max_df"]),
    "tfidf_char__min_df": make_range(best_rnd_params["features__tfidf_char__min_df"]),
    "tfidf_char__max_features": make_range(
        best_rnd_params["tfidf_char__max_features"], cast_int=True
    ),
}
grid_tfidf_params_char

In [ ]:
grid_tfidf_search_char = GridSearchCV(
    Pipeline([
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
        ("clf", LinearSVC(
            C=best_rnd_params["clf__C"],
            random_state=int(os.getenv("RANDOM_SEED", 880055535)),
        )),
    ]),
    grid_tfidf_params_char,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

In [ ]:
grid_tfidf_search_char.fit(texts, y_train)

In [ ]:
best_tfidf_char = grid_tfidf_search_char.best_params_
best_tfidf_char

In [ ]:
wandb_log_search_results(grid_tfidf_search_word, "hyperparams_grid_search_char", "hyperparams_grid_search")

## SVD

In [ ]:
grid_svd_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
grid_merged_params_for_svd = {
    **{f"features__{k}": v for k, v in best_tfidf_word.items()},
    **{f"features__{k}": v for k, v in best_tfidf_char.items()},
}
print(grid_merged_params_for_svd)
grid_svd_pipeline.set_params(**grid_merged_params_for_svd)

In [ ]:
grid_svd_params = {
    "svd__n_components": make_range(best_rnd_params["svd__n_components"], pct=0.1, cast_int=True),
}
grid_svd_params

In [ ]:
grid_svd_search = GridSearchCV(
    grid_svd_pipeline,
    grid_svd_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

grid_svd_search.fit(texts, y_train)

In [ ]:
best_svd = grid_svd_search.best_params_
best_svd_score = grid_svd_search.best_score_

In [ ]:
wandb_log_search_results(grid_tfidf_search_word, "hyperparams_grid_search_svd", "hyperparams_grid_search")

## CLF

In [ ]:
grid_clf_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")),
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))),
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))),
])
grid_merged_params_for_clf = {
    **{f"features__{k}": v for k, v in best_tfidf_word.items()},
    **{f"features__{k}": v for k, v in best_tfidf_char.items()},
    **{f"svd__{k}": v for k, v in best_svd.items()},
}
print(grid_merged_params_for_clf)
grid_clf_pipeline.set_params(**grid_merged_params_for_clf)

In [ ]:
grid_clf_params = {
    "clf__C": make_range(best_rnd_params["clf__C"], pct=0.1),
}
grid_clf_params

In [ ]:
grid_clf_search = GridSearchCV(
    grid_clf_pipeline,
    grid_clf_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

In [ ]:
grid_clf_search.fit(texts, y_train)

In [ ]:
grid_search = grid_clf_search
grid_params = grid_clf_params

In [ ]:
wandb_log_search_results(grid_tfidf_search_word, "hyperparams_grid_search_clf", "hyperparams_grid_search")

In [ ]:
wandb_save_model(
    rnd_clf_search.best_estimator_,
    os.path.join(MODEL_DIR, "best_model_after_grid_search"),
    )